# Feature Extraction

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc


root = r"C:\Users\MHMD RAGAB\Downloads\DATA\New folder\\"

In [2]:
aisles = pd.read_csv(root + 'aisles.csv')

departments = pd.read_csv(root + 'departments.csv')

orders = pd.read_csv(root + 'orders.csv', 
                 dtype={
                        'order_id': np.int32,
                        'user_id': np.int64,
                        'eval_set': 'category',
                        'order_number': np.int16,
                        'order_dow': np.int8,
                        'order_hour_of_day': np.int8,
                        'days_since_prior_order': np.float32})

order_products_prior = pd.read_csv(root + 'order_products__prior.csv', 
                                 dtype={
                                        'order_id': np.int32,
                                        'product_id': np.uint16,
                                        'add_to_cart_order': np.int16,
                                        'reordered': np.int8})

order_products_train = pd.read_csv(root + 'order_products__train.csv', 
                                 dtype={
                                        'order_id': np.int32,
                                        'product_id': np.uint16,
                                        'add_to_cart_order': np.int16,
                                        'reordered': np.int8})
products = pd.read_csv(root + 'products.csv')

### Preparing Data

In [3]:
prior_df = order_products_prior.merge(orders, on ='order_id', how='inner')
prior_df = prior_df.merge(products, on = 'product_id', how = 'left')
prior_df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13


## Features creation

Calculating how many times a user buy the product

In [4]:
prior_df['user_buy_product_times'] = prior_df.groupby(['user_id', 'product_id']).cumcount() + 1
prior_df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,user_buy_product_times
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,1
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,1
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,1
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,1
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,1


### Product level features 

(1) Product's average add-to-cart-order

(2) Total times the product was ordered

(3) Total times the product was reordered

(4) Reorder percentage of a product

(5) Total unique users of a product

(6) Is the product Organic?

(7) Percentage of users that buy the product second time

In [5]:
agg_dict1 = {'add_to_cart_order' : {'mean_add_to_cart_order':'mean'}, 
           'reordered' : {'total_orders':'count', 'total_reorders':'sum', 'reorder_percentage':'mean'},
           'user_id': {'unique_users' :lambda x: x.nunique()},
           'user_buy_product_times': {'order_first_time_total_cnt' : lambda x: sum(x==1), 
                                      'order_second_time_total_cnt' :lambda x: sum(x==2)},
           'product_name':{'is_organic': lambda x: 1 if 'Organic' in x else 0}}

In [6]:
import pandas as pd

# 👇 مثال: لو prior_df هو الداتا الأساسية
# prior_df = order_products__prior.merge(orders, on='order_id')

# 🧠 Product Features (صح 100% بدون nested dict)
prod_feats1 = prior_df.groupby('product_id').agg(
    total_orders=('order_id', 'count'),
    total_reorders=('reordered', 'sum'),
    unique_users=('user_id', 'nunique')
).reset_index()

# =========================
# 🧠 إضافة Features مهمة تانية
# =========================

prod_feats2 = prior_df.groupby('product_id').agg(
    avg_add_to_cart_position=('add_to_cart_order', 'mean'),
    max_add_to_cart_position=('add_to_cart_order', 'max')
).reset_index()

# =========================
# 🔗 دمج كل الـ product features
# =========================

product_features = prod_feats1.merge(prod_feats2, on='product_id')

# =========================
# 👀 عرض النتيجة
# =========================

print(product_features.head())
print(product_features.shape)

   product_id  total_orders  total_reorders  unique_users  \
0           1          1852            1136           716   
1           2            90              12            78   
2           3           277             203            74   
3           4           329             147           182   
4           5            15               9             6   

   avg_add_to_cart_position  max_add_to_cart_position  
0                  5.801836                        74  
1                  9.888889                        45  
2                  6.415162                        41  
3                  9.507599                        36  
4                  6.466667                        16  
(49677, 6)


In [7]:
prod_feats1 = prod_feats1.merge(
    prior_df[prior_df['order_number'] == 1].groupby('product_id').size().reset_index(name='first_time_cnt'),
    on='product_id',
    how='left'
)

prod_feats1 = prod_feats1.merge(
    prior_df[prior_df['order_number'] == 2].groupby('product_id').size().reset_index(name='second_time_cnt'),
    on='product_id',
    how='left'
)

prod_feats1['second_time_percent'] = (
    prod_feats1['second_time_cnt'] / prod_feats1['first_time_cnt']
).fillna(0)

### Aisle and department features

(8) Reorder percentage, Total orders and reorders of a product  aisle

(9) Mean and std of aisle add-to-cart-order

(10) Aisle unique users

In [8]:
agg_dict2 = {'add_to_cart_order' : {'aisle_mean_add_to_cart_order':'mean',
                                   'aisle_std_add_to_cart_order':'std'}, 
           'reordered' : {'aisle_total_orders':'count', 'aisle_total_reorders':'sum', 'aisle_reorder_percentage':'mean'},
           'user_id': {'aisle_unique_users' :lambda x: x.nunique()}}

In [9]:
aisle_feats = prior_df.groupby('aisle_id').agg(
    total_orders=('order_id', 'count'),
    total_reorders=('reordered', 'sum'),
    unique_users=('user_id', 'nunique')
).reset_index()

**features**

(10) Reorder percentage, Total orders and reorders of a product department

(11) Mean and std of department add-to-cart-order

(12) Department unique users

In [10]:
agg_dict3 = {'add_to_cart_order' : {'department_mean_add_to_cart_order':'mean',
                                   'department_std_add_to_cart_order':'std'}, 
           'reordered' : {'department_total_orders':'count', 'department_total_reorders':'sum',
                          'department_reorder_percentage':'mean'},
           'user_id': {'department_unique_users' :lambda x: x.nunique()}}

In [11]:
dpt_feats = prior_df.groupby('department_id').agg(
    total_orders=('order_id', 'count'),
    total_reorders=('reordered', 'sum'),
    unique_users=('user_id', 'nunique')
).reset_index()

**features**

(13) Binary encoding of aisle feature

(14) Binary encoding of department feature

In [12]:
prod_feats1 = prod_feats1.merge(products, on = 'product_id', how = 'left')
prod_feats1 = prod_feats1.merge(aisle_feats, on = 'aisle_id', how = 'left')
prod_feats1 = prod_feats1.merge(aisles, on = 'aisle_id', how = 'left')
prod_feats1 = prod_feats1.merge(dpt_feats, on = 'department_id', how = 'left')
prod_feats1 = prod_feats1.merge(departments, on = 'department_id', how = 'left')
prod_feats1.head()

,product_id,total_orders_x,total_reorders_x,unique_users_x,first_time_cnt,second_time_cnt,second_time_percent,product_name,aisle_id,department_id,total_orders_y,total_reorders_y,unique_users_y,aisle,total_orders,total_reorders,unique_users,department
0,1,1852,1136,716,98.0,103.0,1.051020,Chocolate Sandwich Cookies,61,19,234065,128431,54202,cookies cakes,2887550,1657973,174219,snacks
1,2,90,12,78,3.0,5.0,1.666667,All-Seasons Salt,104,13,212092,32321,76402,spices seasonings,1875577,650301,172755,pantry
2,3,277,203,74,17.0,19.0,1.117647,Robust Golden Unsweetened Oolong Tea,94,7,249341,131556,53197,tea,2690129,1757892,172795,beverages
3,4,329,147,182,39.0,33.0,0.846154,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,390299,217262,58749,frozen meals,2236432,1211890,163233,frozen
4,5,15,9,6,1.0,NaN,0.000000,Green Chile Anytime Sauce,5,13,62510,17542,32312,marinades meat preparation,1875577,650301,172755,pantry


In [13]:
prod_feats1.drop(['product_name', 'aisle_id', 'department_id'], axis = 1, inplace = True)
prod_feats1.head()

,product_id,total_orders_x,total_reorders_x,unique_users_x,first_time_cnt,second_time_cnt,second_time_percent,total_orders_y,total_reorders_y,unique_users_y,aisle,total_orders,total_reorders,unique_users,department
0,1,1852,1136,716,98.0,103.0,1.051020,234065,128431,54202,cookies cakes,2887550,1657973,174219,snacks
1,2,90,12,78,3.0,5.0,1.666667,212092,32321,76402,spices seasonings,1875577,650301,172755,pantry
2,3,277,203,74,17.0,19.0,1.117647,249341,131556,53197,tea,2690129,1757892,172795,beverages
3,4,329,147,182,39.0,33.0,0.846154,390299,217262,58749,frozen meals,2236432,1211890,163233,frozen
4,5,15,9,6,1.0,NaN,0.000000,62510,17542,32312,marinades meat preparation,1875577,650301,172755,pantry


In [14]:
prod_feats1.shape

(49677, 15)

In [15]:
prod_feats1.dtypes

product_id              uint16
total_orders_x           int64
total_reorders_x         int64
unique_users_x           int64
first_time_cnt         float64
second_time_cnt        float64
second_time_percent    float64
total_orders_y           int64
total_reorders_y         int64
unique_users_y           int64
aisle                   object
total_orders             int64
total_reorders           int64
unique_users             int64
department              object
dtype: object

In [16]:
# pip install category_encoders

In [17]:
import category_encoders as ce

encoder = ce.BinaryEncoder(
    cols=['aisle', 'department'],
    return_df=True
)

In [18]:
prod_feats1 = encoder.fit_transform(prod_feats1)
prod_feats1.head()

,product_id,total_orders_x,total_reorders_x,unique_users_x,first_time_cnt,second_time_cnt,second_time_percent,total_orders_y,total_reorders_y,unique_users_y,...,aisle_6,aisle_7,total_orders,total_reorders,unique_users,department_0,department_1,department_2,department_3,department_4
0,1,1852,1136,716,98.0,103.0,1.051020,234065,128431,54202,...,0,1,2887550,1657973,174219,0,0,0,0,1
1,2,90,12,78,3.0,5.0,1.666667,212092,32321,76402,...,1,0,1875577,650301,172755,0,0,0,1,0
2,3,277,203,74,17.0,19.0,1.117647,249341,131556,53197,...,1,1,2690129,1757892,172795,0,0,0,1,1
3,4,329,147,182,39.0,33.0,0.846154,390299,217262,58749,...,0,0,2236432,1211890,163233,0,0,1,0,0
4,5,15,9,6,1.0,NaN,0.000000,62510,17542,32312,...,0,1,1875577,650301,172755,0,0,0,1,0


In [19]:
prod_feats1.shape

(49677, 26)

In [20]:
prod_feats1.columns

Index(['product_id', 'total_orders_x', 'total_reorders_x', 'unique_users_x',
       'first_time_cnt', 'second_time_cnt', 'second_time_percent',
       'total_orders_y', 'total_reorders_y', 'unique_users_y', 'aisle_0',
       'aisle_1', 'aisle_2', 'aisle_3', 'aisle_4', 'aisle_5', 'aisle_6',
       'aisle_7', 'total_orders', 'total_reorders', 'unique_users',
       'department_0', 'department_1', 'department_2', 'department_3',
       'department_4'],
      dtype='object')

In [21]:
prod_feats1.isnull().any().any()

True

In [22]:
# free some memory
del aisle_feats, dpt_feats, aisles, departments
gc.collect()

0

### User level features

(15) User's average and std day-of-week of order

(16) User's average and std hour-of-day of order

(17) User's average and std days-since-prior-order

(18) Total orders by a user

(19) Total products user has bought

(20) Total unique products user has bought

(21) user's total reordered products

(22) User's overall reorder percentage

In [23]:
prior_df.isnull().any()

order_id                  False
product_id                False
add_to_cart_order         False
reordered                 False
user_id                   False
eval_set                  False
order_number              False
order_dow                 False
order_hour_of_day         False
days_since_prior_order     True
product_name              False
aisle_id                  False
department_id             False
user_buy_product_times    False
dtype: bool

In [24]:
# when no prior order, the value is null. Imputing as 0
prior_df.days_since_prior_order = prior_df.days_since_prior_order.fillna(0)

In [25]:
prior_df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,user_buy_product_times
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,1
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,1
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,1
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,1
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,1


In [26]:
agg_dict4 = {'order_dow': {'avg_dow':'mean', 'std_dow':'std'},
           'order_hour_of_day': {'avg_doh':'mean', 'std_doh':'std'},
           'days_since_prior_order': {'avg_since_order':'mean', 'std_since_order':'std'},
           'order_number': {'total_orders_by_user': lambda x: x.nunique()},
           'product_id': {'total_products_by_user': 'count',
                         'total_unique_product_by_user': lambda x: x.nunique()},
           'reordered': {'total_reorders_by_user':'sum', 
                        'reorder_propotion_by_user':'mean'}}

In [27]:
user_feats = prior_df.groupby('user_id').agg(
    total_orders=('order_id', 'count'),
    total_products=('product_id', 'count'),
    unique_products=('product_id', 'nunique'),
    total_reorders=('reordered', 'sum')
).reset_index()

**features**

(23) Average order size of a user

(24) User's mean of reordered items of all orders

In [28]:
user_feats2 = prior_df.groupby(['user_id', 'order_number']).agg(
    average_order_size=('reordered', 'count'),
    reorder_in_order=('reordered', 'mean')
).reset_index()

In [29]:
user_feats3 = user_feats2.groupby('user_id').agg({'average_order_size' : 'mean', 
                                   'reorder_in_order':'mean'})
user_feats3 = user_feats3.reset_index()
user_feats3.head()

,user_id,average_order_size,reorder_in_order
0,1,5.900000,0.705833
1,2,13.928571,0.447961
2,3,7.333333,0.658817
3,4,3.600000,0.028571
4,5,9.250000,0.377778


In [30]:
user_feats = user_feats.merge(user_feats3, on = 'user_id', how = 'left')
user_feats.head()

,user_id,total_orders,total_products,unique_products,total_reorders,average_order_size,reorder_in_order
0,1,59,59,18,41,5.900000,0.705833
1,2,195,195,102,93,13.928571,0.447961
2,3,88,88,33,55,7.333333,0.658817
3,4,18,18,17,1,3.600000,0.028571
4,5,37,37,23,14,9.250000,0.377778


**features**

(25) Percentage of reordered itmes in user's last three orders

(26) Total orders in user's last three orders

Last 3 orders of a user

In [ ]:
last_three_orders = user_feats2.groupby('user_id')['order_number'].nlargest(3).reset_index()
last_three_orders.head()

In [ ]:
last_three_orders = user_feats2.merge(last_three_orders, on = ['user_id', 'order_number'], how = 'inner')
last_three_orders.head()

In [ ]:
last_three_orders['rank'] = last_three_orders.groupby("user_id")["order_number"].rank("dense", ascending=True)

In [ ]:
last_order_feats = last_three_orders.pivot_table(index = 'user_id', columns = ['rank'], \
                                                 values=['average_order_size', 'reorder_in_order']).\
                                                reset_index(drop = False)
last_order_feats.columns = ['user_id','orders_3', 'orders_2', 'orders_1', 'reorder_3', 'reorder_2', 'reorder_1']
last_order_feats.head()

In [ ]:
user_feats = user_feats.merge(last_order_feats, on = 'user_id', how = 'left')
user_feats.head()

### User and Product level features

(27) User's avg add-to-cart-order for a product

(28) User's avg days_since_prior_order for a product

(29) User's product total orders, reorders and reorders percentage

(30) User's order number when the product was bought last

In [ ]:
agg_dict6 = {'reordered': {'total_product_orders_by_user':'count', 
                          'total_product_reorders_by_user':'sum',
                          'user_product_reorder_percentage': 'mean'},
            'add_to_cart_order': {'avg_add_to_cart_by_user':'mean'},
            'days_since_prior_order': {'avg_days_since_last_bought':'mean'},
            'order_number': {'last_ordered_in':'max'}}

In [ ]:
user_product_feats = prior_df.groupby(['user_id', 'product_id']).agg(
    total_orders=('order_id', 'count'),
    total_reorders=('reordered', 'sum'),
    last_order_number=('order_number', 'max'),
    avg_cart_position=('add_to_cart_order', 'mean')
).reset_index()

**features**

(31) User's product purchase history of last three orders

In [ ]:
last_three_orders.head()

In [ ]:
last_orders = prior_df.merge(last_three_orders, on = ['user_id', 'order_number'], how = 'inner')
last_orders.head()

In [ ]:
last_orders['rank'] = last_orders.groupby(['user_id', 'product_id'])['order_number'].rank("dense", ascending=True)

In [ ]:
product_purchase_history = last_orders.pivot_table(index = ['user_id', 'product_id'],\
                                                   columns='rank', values = 'reordered').reset_index()
product_purchase_history.columns = ['user_id', 'product_id', 'is_reorder_3', 'is_reorder_2', 'is_reorder_1']
product_purchase_history.fillna(0, inplace = True)
product_purchase_history.head()

In [ ]:
user_product_feats = user_product_feats.merge(product_purchase_history, on=['user_id', 'product_id'], how = 'left')
user_product_feats.head()

In [ ]:
user_product_feats.isnull().sum()

In [ ]:
user_product_feats.fillna(0, inplace = True)

## Saving all features

In [ ]:
prod_feats1.to_pickle(root + 'product_features.pkl')
user_feats.to_pickle(root +'user_features.pkl')
user_product_feats.to_pickle(root +'user_product_features.pkl')

In [ ]:
df = pd.read_pickle(root +'product_features.pkl')
df.head()

In [ ]:
df = pd.read_pickle(root+'user_features.pkl')
df.head()

In [ ]:
df = pd.read_pickle(root + 'user_product_features.pkl')
df.head()